# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

## 1. Raw 데이터 품질 확인
- total_users = 677,085
- dup_id = 0, null_id = 0
- null_created_at = 0 → 가입시각 전부 존재. "첫 수신까지 걸린 시간" 분모 걱정 없음
- null_group_id = 3, null_gender = 2 → 거의 0. 결측이 사실상 없음

In [ ]:
%%bigquery df_user_qc --project your-gcp-project
-- stg_users용 Raw 품질 확인 (타입은 아래 별도 셀)
SELECT
  COUNT(*)                              AS total_users,
  COUNT(*) - COUNT(DISTINCT id)         AS dup_id,          -- 0이어야
  COUNTIF(id IS NULL)                   AS null_id,          -- 0이어야
  COUNTIF(created_at IS NULL)           AS null_created_at,  -- 가입시각 결측
  COUNTIF(group_id IS NULL)             AS null_group_id,    -- 학급 결측(축 B 영향)
  COUNTIF(gender IS NULL)               AS null_gender       -- 성별 결측
FROM `your-gcp-project.raw.raw_user`;

In [ ]:
df_user_qc

## 2. 타입
- created_at = TIMESTAMP → KST 변환 대상
- 스테이징 시 (VIEW 생성 시) 변환

In [ ]:
%%bigquery df_user_types --project your-gcp-project
SELECT column_name, data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'raw_user'
  AND column_name IN ('id','created_at','group_id','gender','point','ban_status','is_superuser','is_staff')
ORDER BY ordinal_position;

In [ ]:
df_user_types

## 3. ban_status 분포
- N이 98.7%로 압도적이고, 나머지 3개가 차단·탈퇴 계열 (1.3%)
- is_valid_user = (ban_status = 'N') 로 정의하는 게 안전

In [ ]:
%%bigquery df_ban --project your-gcp-project
SELECT ban_status, COUNT(*) AS cnt,
       ROUND(COUNT(*)/SUM(COUNT(*)) OVER ()*100,1) AS pct
FROM `your-gcp-project.raw.raw_user`
GROUP BY ban_status ORDER BY cnt DESC;

In [ ]:
df_ban

## 4. Block & Hide

In [ ]:
%%bigquery df_list_sample --project your-gcp-project
-- block/hide 컬럼 실제 값 형태 확인 (NULL / '' / '[]' / '1,2,3' 중 뭔지)
SELECT
  block_user_id_list,
  hide_user_id_list
FROM `your-gcp-project.raw.raw_user`
WHERE block_user_id_list IS NOT NULL
   OR hide_user_id_list IS NOT NULL
LIMIT 20;

In [ ]:
df_list_sample

In [ ]:
%%bigquery df_block_hide --project your-gcp-project
WITH flags AS (
  SELECT
    (block_user_id_list IS NOT NULL
       AND TRIM(block_user_id_list) NOT IN ('', '[]')) AS has_block,
    (hide_user_id_list IS NOT NULL
       AND TRIM(hide_user_id_list)  NOT IN ('', '[]')) AS has_hide
  FROM `your-gcp-project.raw.raw_user`
)
SELECT
  COUNT(*)                                  AS total_users,
  COUNTIF(has_block)                        AS users_with_block,
  COUNTIF(has_hide)                         AS users_with_hide,
  COUNTIF(has_block OR has_hide)            AS users_with_either,
  ROUND(COUNTIF(has_block)/COUNT(*)*100,1)  AS block_pct,
  ROUND(COUNTIF(has_hide)/COUNT(*)*100,1)   AS hide_pct
FROM flags;

In [ ]:
df_block_hide

## 5. 실제 결측값 재확인

### group_id / gender가 NULL인 유저가 운영자 계정인지 확인

In [ ]:
%%bigquery df_null_check --project your-gcp-project
-- group_id / gender가 NULL인 유저가 운영자 계정인지 확인
SELECT
  (is_superuser OR is_staff) AS is_staff_account,
  COUNT(*)                   AS cnt,
  COUNTIF(group_id IS NULL)  AS null_group,
  COUNTIF(gender IS NULL)    AS null_gender
FROM `your-gcp-project.raw.raw_user`
GROUP BY is_staff_account;

In [ ]:
df_null_check

### 실유저만 놓고 품질 재확인
- 물리적 삭제 X
- where 필터로 거르기

In [ ]:
%%bigquery df_valid_qc --project your-gcp-project
-- 운영자 아닌 실유저만 놓고 품질 재확인
WITH real_users AS (
  SELECT *
  FROM `your-gcp-project.raw.raw_user`
  WHERE NOT (is_superuser OR is_staff)   -- 운영자 제외
)
SELECT
  COUNT(*)                        AS real_user_cnt,
  COUNT(*) - COUNT(DISTINCT id)   AS dup_id,          -- 중복
  COUNTIF(id IS NULL)             AS null_id,
  COUNTIF(created_at IS NULL)     AS null_created_at,
  COUNTIF(group_id IS NULL)       AS null_group_id,   -- 이게 0이면 네 말이 맞음
  COUNTIF(gender IS NULL)         AS null_gender      -- 이것도
FROM real_users;

In [ ]:
df_valid_qc

### 인코딩

In [ ]:
%%bigquery df_encoding --project your-gcp-project
SELECT
    id AS user_id,
    gender,
    point,
    CASE
        WHEN block_user_id_list = '[]' THEN 0
        ELSE ARRAY_LENGTH(JSON_VALUE_ARRAY(block_user_id_list))
    END AS blocked_user_count,
    CASE
        WHEN hide_user_id_list = '[]' THEN 0
        ELSE ARRAY_LENGTH(JSON_VALUE_ARRAY(hide_user_id_list))
    END AS hidden_user_count
FROM your-gcp-project.raw.raw_user;

In [ ]:
df_encoding

## 6. 스냅샷 컬럼 확인
- 덤프 시점의 순간값이라 언제인지 시점 알 수 없음, 히스토리 파악 불가
- 전처리라기보다 음수·이상치 점검
- `max_point` = 885,000,006
- `max_report` = 253, `max_pending_votes` = 3,352
> Stage에선 값 그대로 두고, EDA 시 활용하기

In [ ]:
%%bigquery df_snapshot_qc --project your-gcp-project
-- 스냅샷 컬럼 이상치·결측 점검 (처리 전 확인용)
SELECT
  -- 음수 체크 (논리적으로 0 이상이어야 정상)
  COUNTIF(point < 0)          AS neg_point,
  COUNTIF(report_count < 0)   AS neg_report,
  COUNTIF(alarm_count < 0)    AS neg_alarm,
  COUNTIF(pending_chat < 0)   AS neg_pending_chat,
  COUNTIF(pending_votes < 0)  AS neg_pending_votes,
  -- NULL 체크
  COUNTIF(point IS NULL)         AS null_point,
  COUNTIF(report_count IS NULL)  AS null_report,
  COUNTIF(is_push_on IS NULL)    AS null_push,
  -- 최댓값 (비현실적으로 큰 값 있는지 확인)
  MAX(point)          AS max_point,
  MAX(report_count)   AS max_report,
  MAX(pending_votes)  AS max_pending_votes
FROM `your-gcp-project.raw.raw_user`;

In [ ]:
df_snapshot_qc

### max_point 유저 정체 확인
- 10명 다 is_staff_account=False, ban_status='N'

In [ ]:
%%bigquery df_outlier --project your-gcp-project
-- 극단적 포인트 보유 유저 상위 10명 정체 확인
SELECT
  id,
  point,
  DATETIME(created_at,'Asia/Seoul') AS signup_at,
  ban_status,
  is_superuser,
  is_staff,
  (is_superuser OR is_staff) AS is_staff_account
FROM `your-gcp-project.raw.raw_user`
ORDER BY point DESC
LIMIT 10;

In [ ]:
df_outlier

In [ ]:
%%bigquery df_outlier_behavior --project your-gcp-project
-- 8.8억 유저(833041)가 실제로 활동하는 유저인지 확인
SELECT
  COUNT(*)                       AS total_pings_received,   -- 선택받은 횟수
  COUNTIF(is_hint_opened)        AS hint_opened,            -- 힌트 연 횟수
  MIN(voted_at)                  AS first_received,
  MAX(voted_at)                  AS last_received
FROM `your-gcp-project.stage.stg_vote`
WHERE receiver_user_id = 833041;

In [ ]:
df_outlier_behavior

- total_pings_received = 0 → 선택받은 적 없음
- hint_opened = 0 → 힌트 연 적 없음
- first_received = NaT → 수신 이력 자체가 없음

## VIEW 생성

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_users` AS
WITH src AS (
  SELECT
    id, created_at, group_id, gender, point,
    friend_id_list, block_user_id_list, hide_user_id_list,
    is_push_on, report_count, alarm_count, pending_chat, pending_votes,
    ban_status, is_superuser, is_staff
  FROM `your-gcp-project.raw.raw_user`
)
SELECT
  -- ── 식별 / 기본 속성 ──
  id                                 AS user_id,
  DATETIME(created_at, 'Asia/Seoul') AS signup_at,      -- 가입시각(KST)
  group_id,                                             -- 학급 (축 B)
  gender,                                               -- 성별

  -- ── 리스트 → 인원수 인코딩 ──
  COALESCE(ARRAY_LENGTH(JSON_EXTRACT_ARRAY(block_user_id_list)), 0) AS block_count,  -- 차단한 인원수
  COALESCE(ARRAY_LENGTH(JSON_EXTRACT_ARRAY(hide_user_id_list)),  0) AS hide_count,   -- 숨김한 인원수
  COALESCE(ARRAY_LENGTH(JSON_EXTRACT_ARRAY(friend_id_list)),     0) AS friend_count, -- 친구 수 (선택)

  -- ── 스냅샷(덤프 시점 현재값) ⚠️ 시계열 분석 금지, "현재 상태"로만 해석 ──
  point,                                                -- 포인트 잔액
  report_count,                                         -- 누적 신고당한 횟수
  alarm_count,                                          -- 미확인 알림 수
  pending_chat,                                         -- 미확인 채팅 수
  pending_votes,                                        -- 미확인 투표 수
  is_push_on,                                           -- 푸시 설정

  -- ── 상태 / 플래그 ──
  ban_status,                                           -- 원본 보존
  (ban_status = 'N')             AS is_valid_user,      -- 정상 유저(ban 아님)
  (is_superuser OR is_staff)     AS is_staff_account    -- 운영자 계정
FROM src;

### 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*)                     AS total_users,       -- 677,085
  COUNTIF(is_valid_user)       AS valid_users,       -- 668,432 (ban N)
  COUNTIF(is_staff_account)    AS staff_accounts,    -- 4
  COUNTIF(signup_at IS NULL)   AS null_signup,       -- 0
  COUNTIF(block_count > 0)     AS users_with_block,  -- 13,043 ← 교차검증
  COUNTIF(hide_count  > 0)     AS users_with_hide,   -- 78,733 ← 교차검증
  MAX(block_count)             AS max_block,
  MAX(hide_count)              AS max_hide
FROM `your-gcp-project.stage.stg_users`;